# Probe 023 launcher (phase C)
Colab is a compute worker only. This driver kernel NEVER imports the model stack -- `run.py` runs as a child process, so no kernel restart is ever needed.

**One-time setup:** create a fine-grained GitHub PAT scoped to this single repository, Contents: Read and write, with an expiry. In Colab: key icon (Secrets) -> add `SCOUT_RESULTS_PAT` -> enable notebook access. The PAT never appears in this notebook or its output.

Results branch (contract-bound): `results/probe-023-349af5ad0b3e`

Per session: run all cells top to bottom. After a disconnect, rerun all cells -- run.py resumes from the bundle on Drive, and the transport cell pushes whatever is new.

In [ ]:
PHASE = 'C'
REPO_URL = 'https://github.com/Moroseui/concept-research-scout.git'
PIN_COMMIT = '62916534a2dcb3b2d74e31ab15af87fc65452145'
RESULTS_BRANCH = 'results/probe-023-349af5ad0b3e'
OUTPUT_DIR = '/content/drive/MyDrive/concept-research-scout-results/023_C'
PHASE_S_DIR = '/content/drive/MyDrive/concept-research-scout-results/023_v2'

In [ ]:
from google.colab import drive, userdata
import os, shutil, time
MP = '/content/drive'
def _mounted(mp):
    try:
        return any(len(l.split()) > 1 and l.split()[1] == mp
                   for l in open('/proc/mounts'))
    except OSError:
        return False
try:
    if os.path.isdir(MP) and not _mounted(MP) and os.listdir(MP):
        stale = f'/content/drive_stale_{int(time.time())}'
        shutil.move(MP, stale)
        print('stale mountpoint residue moved to', stale,
              '(a crashed FUSE mount left a corpse on a surviving VM)')
except OSError as e:
    print('could not inspect/move mountpoint residue:', e,
          '-- if the mount below fails, Runtime > Disconnect and delete runtime')
drive.mount(MP, force_remount=True)
GH_PAT = userdata.get('SCOUT_RESULTS_PAT')  # never printed
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # inherited by the run.py child; never printed

In [ ]:
%cd /content
!rm -rf /content/scout-repo
!git clone {REPO_URL} /content/scout-repo
%cd /content/scout-repo
!git checkout {PIN_COMMIT}

In [ ]:
!pip install -q -r probes/023/requirements.txt

In [ ]:
# --- Generated staging: Zenodo record 16813698 (Drive-persistent; a pin NEVER silently re-resolves) ---
import os, json, urllib.request
STAGE = '/content/drive/MyDrive/staging-16731717'
RECORD_JSON = STAGE + '/zenodo_record.json'
DATA_DIR = STAGE + '/extracted'
os.makedirs(STAGE, exist_ok=True)
_need = not os.path.exists(RECORD_JSON)
if not _need:
    _have = str(json.load(open(RECORD_JSON)).get('id'))
    if _have != '16813698':
        print('re-pinning to declared record 16813698 (found', _have + ');',
              'a runtime drift here is a reproducibility bug -- investigate before trusting old outputs')
        _need = True
if _need:
    with urllib.request.urlopen('https://zenodo.org/api/records/16813698') as r:
        rec = json.load(r)
    assert str(rec['id']) == '16813698', 'server returned a different record than the declared pin'
    json.dump(rec, open(RECORD_JSON, 'w'), indent=2)
rec = json.load(open(RECORD_JSON))
_a = [f for f in rec['files'] if f['key'].endswith('.7z')]
assert len(_a) == 1, _a
ARCHIVE = STAGE + '/' + _a[0]['key']
ARCHIVE_URL = _a[0]['links']['self']
print('pinned record', rec['id'], _a[0]['key'], round(_a[0]['size']/1e9, 1), 'GB')

In [ ]:
!wget -c -O "{ARCHIVE}" "{ARCHIVE_URL}"

In [ ]:
# --- Localize heavy inputs: the Drive FUSE mount is unreliable under
# deep small-file trees (three recorded casualties on idea 023). One
# bounded FUSE read copies the archive to local SSD; extraction,
# digests, and the census then run on local disk. Outputs stay on
# Drive for persistence.
import shutil
LOCAL = '/content/work'
os.makedirs(LOCAL, exist_ok=True)
ARCHIVE_LOCAL = LOCAL + '/' + os.path.basename(ARCHIVE)
LOCAL_DATA = LOCAL + '/extracted'
import hashlib
_name = os.path.basename(ARCHIVE)
_entries = [f for f in rec['files'] if f.get('key') == _name]
assert len(_entries) == 1, ('record must contain exactly one entry named ' + _name)
_ck = _entries[0].get('checksum', '')
if not _ck.startswith('md5:'):
    raise SystemExit('driver configuration error: pinned record supplies no '
                     'md5 for ' + _name + '; refusing a 99 GB staging pass '
                     'without a transport checksum')
EXPECT_MD5 = _ck.split(':', 1)[1]
EXPECT_SIZE = _entries[0]['size']
def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()
_done = False
if os.path.exists(ARCHIVE_LOCAL):
    print('verifying existing local archive md5 (~4 min on scratch)...')
    if os.path.getsize(ARCHIVE_LOCAL) == EXPECT_SIZE and _md5(ARCHIVE_LOCAL) == EXPECT_MD5:
        _done = True
    else:
        print('existing local archive fails integrity; removing')
        os.remove(ARCHIVE_LOCAL)
if not _done:
    for _attempt in (1, 2):
        _part = ARCHIVE_LOCAL + '.part'
        print('copying archive Drive -> local scratch (attempt', _attempt,
              'of 2; expect 20-60 min)...')
        shutil.copyfile(ARCHIVE, _part)
        print('verifying transferred bytes md5 (~4 min)...')
        if os.path.getsize(_part) == EXPECT_SIZE and _md5(_part) == EXPECT_MD5:
            os.replace(_part, ARCHIVE_LOCAL)  # atomic promotion; no size-exact liars
            _done = True
            break
        print('transfer failed integrity (suspected DriveFS/FUSE read-path '
              'corruption; mechanism not asserted); discarding partial')
        os.remove(_part)
if not _done:
    raise SystemExit('FUSE_LOCALIZATION_INTEGRITY_FAILURE: two Drive->local '
                     'transfers failed md5. The stored Drive master is NOT '
                     'judged from this path (it would be the same suspect '
                     'witness). Sanctioned next step: origin_direct staging '
                     'from the pinned Zenodo record.')
print('local archive verified: md5', EXPECT_MD5)
print('local archive bytes:', os.path.getsize(ARCHIVE_LOCAL))

In [ ]:
SUFFIXES = ['_space-ncct_cbf.nii.gz', '_space-ncct_cbv.nii.gz', '_space-ncct_mtt.nii.gz', '_space-ncct_tmax.nii.gz', '_lesion-msk.nii.gz', '_ncct.nii.gz']
if not os.path.isdir(LOCAL_DATA):
    !apt-get -qq install -y p7zip-full
    _inc = ' '.join('-ir!*' + x for x in SUFFIXES)
    !7z x "{ARCHIVE_LOCAL}" -o"{LOCAL_DATA}" {_inc} -y
_n = sum(len(f) for _, _, f in os.walk(LOCAL_DATA))
print('extracted files (local):', _n)
assert _n >= 800, 'extraction incomplete -- refusing to reach the census'

In [ ]:
# Console (incl. any crash traceback) persists to Drive; refresh-proof.
!mkdir -p {OUTPUT_DIR}
!python probes/023/run.py --phase {PHASE} --output-dir {OUTPUT_DIR} --data-dir {LOCAL_DATA} --archive-file {ARCHIVE_LOCAL} --record-json {RECORD_JSON} --phase-s-dir {PHASE_S_DIR} 2>&1 | tee -a {OUTPUT_DIR}/driver_console.log

In [ ]:
# E1 transport: mirror the bundle onto the contract-bound results
# branch. ORDER MATTERS: check out the branch FIRST, then overlay the
# bundle (copy-then-checkout fails after session 1: git refuses to
# overwrite untracked files the branch already tracks). The PAT rides
# in a header, never in argv or output.
import shutil, subprocess, pathlib, base64, datetime
repo = pathlib.Path('/content/scout-repo')
dest = repo / 'probes/023/results_v2'
def git(*a, **k):
    r = subprocess.run(['git', *a], cwd=repo, capture_output=True, text=True, **k)
    if r.returncode: raise SystemExit(f'git {a[0]} failed: {r.stderr[-400:]}')
    return r.stdout
git('config', 'user.email', 'colab-runner@scout.local')
git('config', 'user.name', 'scout colab runner')
auth = base64.b64encode(f'x-access-token:{GH_PAT}'.encode()).decode()
hdr = f'http.extraheader=AUTHORIZATION: basic {auth}'
if subprocess.run(['git', '-c', hdr, 'fetch', 'origin', RESULTS_BRANCH], cwd=repo, capture_output=True).returncode == 0:
    git('checkout', '-B', RESULTS_BRANCH, f'origin/{RESULTS_BRANCH}')
else:
    git('checkout', '-B', RESULTS_BRANCH, PIN_COMMIT)
if dest.exists(): shutil.rmtree(dest)
shutil.copytree(OUTPUT_DIR, dest)
git('add', '-f', 'probes/023/results_v2')
stamp = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
subprocess.run(['git', 'commit', '-m', f'session results {stamp}'], cwd=repo, capture_output=True)
git('-c', hdr, 'push', 'origin', RESULTS_BRANCH)
print('pushed', RESULTS_BRANCH)

When `run.py` reports the study complete, the results-validate workflow on the pushed branch verifies the bundle and opens the record-result PR. Merging that PR is the human gate.